# FIFA World Cup 2026 — Predicting Every Match, Factor by Factor

A transparent, reproducible model that predicts every FIFA World Cup 2026 match — all
72 group games explicitly, plus a 10,000-run Monte-Carlo of the knockouts — **and**
explains the result through a deep-dive on the forces that actually decide tournaments:
star power, squad experience, league pedigree, attacking/defensive balance and
World-Cup history. The measurable factors are folded back into the model so they move
the probabilities, not just the commentary.

**Pipeline**
1. Load four public Kaggle datasets (all real, externally sourced).
2. EDA on 150+ years of internationals.
3. Build **World Football Elo** from scratch (leak-free).
4. Engineer team features + **factor deep-dives** (Sections 5–6).
5. Calibrate **Poisson** goals + benchmark **Logistic / Random Forest / XGBoost**.
6. **Fold factors into an adjusted rating** (Section 8) → predictions shift.
7. Predict 72 group matches; simulate the tournament 10,000×.

**Data provenance — every input is a public Kaggle dataset**

| Dataset | Kaggle slug | Files |
|---|---|---|
| International football results 1872–present | `martj42/international-football-results-from-1872-to-2017` | results, shootouts, goalscorers |
| Football Data from Transfermarkt | `davidcariboo/player-scores` | national_teams, players |
| FC 26 Player Stats (EA Sports FC 26) | `talhademirezen/fc-26-player-stats` | fc_26_players_stats |
| Football Players Stats 2025-2026 (FBref) | `hubertsidorowicz/football-players-stats-2025-2026` | *(optional EDA)* |

> The 2026 group draw is hard-coded from the **official FIFA Final Draw** (5 Dec 2025),
> and the **26-man squads** from Wikipedia's official squad lists (Section 2.2).
> Coach quality, weather and injuries are discussed honestly in Section 12 — they are
> **not** in the data, so they are not invented as model features.

---
## 1 · Setup & Loading
### 1.1 Imports

In [ ]:
import os, warnings, itertools, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .25, "font.size": 10})
RNG = np.random.default_rng(42)
print("Libraries ready.")

### 1.2 Robust path resolver
On Kaggle each dataset mounts read-only at `/kaggle/input/<slug>/`. The helper finds
each file there first, then falls back to the working directory, so the **same
notebook runs unchanged on Kaggle or locally**.

> **On Kaggle:** click **Add Input** and attach the four datasets above.

In [ ]:
KAGGLE_DIRS = {
    "results": "/kaggle/input/international-football-results-from-1872-to-2017",
    "tmarkt":  "/kaggle/input/player-scores",
    "fc26":    "/kaggle/input/fc-26-player-stats",
    "fbref":   "/kaggle/input/football-players-stats-2025-2026",
}
def find_file(filename):
    cands = [os.path.join(d, filename) for d in KAGGLE_DIRS.values()]
    cands += [filename, os.path.join(".", filename)]
    if os.path.isdir("/kaggle/input"):
        for root,_d,files in os.walk("/kaggle/input"):
            if filename in files: cands.append(os.path.join(root, filename))
    for c in cands:
        if os.path.exists(c): return c
    raise FileNotFoundError(f"Could not find {filename}. On Kaggle use 'Add Input'.")

for f in ["results.csv","fc_26_players_stats.csv","national_teams.csv","players.csv"]:
    print(f"{f:30s} -> {find_file(f)}")

### 1.3 Load & clean (freeze at the 11 June 2026 kickoff — no leakage)

In [ ]:
results     = pd.read_csv(find_file("results.csv"),     parse_dates=["date"])
shootouts   = pd.read_csv(find_file("shootouts.csv"),   parse_dates=["date"])
goalscorers = pd.read_csv(find_file("goalscorers.csv"), parse_dates=["date"])
fc26        = pd.read_csv(find_file("fc_26_players_stats.csv"))
nat_teams   = pd.read_csv(find_file("national_teams.csv"))
players_tm  = pd.read_csv(find_file("players.csv"))

# The results file may include future/scheduled fixtures (blank scores) and even early
# WC-2026 games. Drop blank scores and keep only matches before kickoff -> true
# PRE-tournament model.
WC2026_START = pd.Timestamp("2026-06-11")
results = results.dropna(subset=["home_score","away_score"])
results = results[results.date < WC2026_START].copy()
results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)
results = results.sort_values("date").reset_index(drop=True)

print(f"results : {len(results):>7,} matches ({results.date.min().date()} -> {results.date.max().date()}) [pre-tournament]")
print(f"fc26    : {len(fc26):>7,} players | players_tm: {len(players_tm):>7,} | national_teams: {len(nat_teams)}")

---
## 2 · The 2026 Tournament Structure
Official FIFA Final Draw (5 Dec 2025). Names normalised to `results.csv`.

In [ ]:
GROUPS = {
 "A": ["Mexico","South Africa","South Korea","Czech Republic"],
 "B": ["Canada","Bosnia and Herzegovina","Qatar","Switzerland"],
 "C": ["Brazil","Morocco","Haiti","Scotland"],
 "D": ["United States","Paraguay","Australia","Turkey"],
 "E": ["Germany","Curaçao","Ivory Coast","Ecuador"],
 "F": ["Netherlands","Japan","Sweden","Tunisia"],
 "G": ["Belgium","Egypt","Iran","New Zealand"],
 "H": ["Spain","Cape Verde","Saudi Arabia","Uruguay"],
 "I": ["France","Senegal","Iraq","Norway"],
 "J": ["Argentina","Algeria","Austria","Jordan"],
 "K": ["Portugal","DR Congo","Uzbekistan","Colombia"],
 "L": ["England","Croatia","Ghana","Panama"],
}
WC_TEAMS = [t for g in GROUPS.values() for t in g]
HOSTS = {"United States","Canada","Mexico"}
CONF = {
 "Mexico":"CONCACAF","United States":"CONCACAF","Canada":"CONCACAF","Panama":"CONCACAF","Curaçao":"CONCACAF","Haiti":"CONCACAF",
 "Brazil":"CONMEBOL","Argentina":"CONMEBOL","Uruguay":"CONMEBOL","Colombia":"CONMEBOL","Paraguay":"CONMEBOL","Ecuador":"CONMEBOL",
 "Spain":"UEFA","France":"UEFA","England":"UEFA","Portugal":"UEFA","Netherlands":"UEFA","Germany":"UEFA","Belgium":"UEFA",
 "Croatia":"UEFA","Switzerland":"UEFA","Austria":"UEFA","Norway":"UEFA","Scotland":"UEFA","Czech Republic":"UEFA",
 "Bosnia and Herzegovina":"UEFA","Sweden":"UEFA","Turkey":"UEFA",
 "Morocco":"CAF","Senegal":"CAF","Egypt":"CAF","Ivory Coast":"CAF","Algeria":"CAF","Tunisia":"CAF","Ghana":"CAF",
 "Cape Verde":"CAF","South Africa":"CAF","DR Congo":"CAF",
 "Japan":"AFC","South Korea":"AFC","Iran":"AFC","Australia":"AFC","Saudi Arabia":"AFC","Qatar":"AFC","Uzbekistan":"AFC",
 "Jordan":"AFC","Iraq":"AFC","New Zealand":"OFC",
}
assert len(WC_TEAMS)==48
print(f"{len(WC_TEAMS)} teams, {len(GROUPS)} groups. Conf:", pd.Series(CONF).loc[WC_TEAMS].value_counts().to_dict())

### 2.1 Name harmonisation (their names → canonical results.csv names)

In [ ]:
FC26_TO_CANON = {"Korea Republic":"South Korea","Côte d'Ivoire":"Ivory Coast",
                 "Holland":"Netherlands","Congo DR":"DR Congo","Cape Verde Islands":"Cape Verde"}
TM_TO_CANON   = {"Korea, South":"South Korea","Bosnia-Herzegovina":"Bosnia and Herzegovina",
                 "Cote d'Ivoire":"Ivory Coast","Congo DR":"DR Congo","Türkiye":"Turkey"}
def canon_fc26(n): return FC26_TO_CANON.get(n,n)
def canon_tm(n):   return TM_TO_CANON.get(n,n)
fc26["canon"]      = fc26["Nation"].map(canon_fc26)
nat_teams["canon"] = nat_teams["country_name"].map(canon_tm)
players_tm["canon"]= players_tm["country_of_citizenship"].map(canon_tm)
print("WC teams missing from FC26 :", sorted(set(WC_TEAMS)-set(fc26.canon)))
print("WC teams missing from TMarkt:", sorted(set(WC_TEAMS)-set(nat_teams.canon)))

### 2.2 The official 26-man squads

Every nation registered a final **26-player squad** with FIFA (deadline 1 June 2026).
Below are all 48 squads — **1,248 players** with their international caps — parsed from
Wikipedia's *2026 FIFA World Cup squads*. The squad-quality features in Section 5 are
built from these **actual** players (not a best-23 estimate). Each entry is
`[player name, international caps]`.

In [ ]:
# Official 2026 WC 26-man squads (name, caps) parsed from Wikipedia
SQUADS_2026 = {'Czech Republic': [['Matěj Kovář', 20], ['David Zima', 25], ['Tomáš Holeš', 41], ['Robin Hranáč', 14], ['Vladimír Coufal', 62], ['Štěpán Chaloupek', 5], ['Ladislav Krejčí', 27], ['Vladimír Darida', 79], ['Adam Hložek', 43], ['Patrik Schick', 53], ['Jan Kuchta', 31], ['Lukáš Červ', 17], ['Mojmír Chytil', 22], ['David Jurásek', 18], ['Pavel Šulc', 21], ['Jindřich Staněk', 14], ['Lukáš Provod', 38], ['Michal Sadílek', 35], ['Tomáš Chorý', 22], ['Jaroslav Zelený', 23], ['David Douděra', 17], ['Tomáš Souček', 90], ['Lukáš Horníček', 1], ['Alexandr Sojka', 2], ['Hugo Sochůrek', 1], ['Denis Višinský', 2]], 'Mexico': [['Raúl Rangel', 14], ['Jorge Sánchez', 59], ['César Montes', 67], ['Edson Álvarez', 98], ['Johan Vásquez', 46], ['Érik Lira', 25], ['Luis Romo', 63], ['Álvaro Fidalgo', 4], ['Raúl Jiménez', 124], ['Alexis Vega', 52], ['Santiago Giménez', 47], ['Carlos Acevedo', 7], ['Guillermo Ochoa', 152], ['Armando González', 7], ['Israel Reyes', 34], ['Julián Quiñones', 22], ['Orbelín Pineda', 92], ['Obed Vargas', 6], ['Gilberto Mora', 8], ['Mateo Chávez', 10], ['César Huerta', 26], ['Guillermo Martínez', 12], ['Jesús Gallardo', 121], ['Luis Chávez', 45], ['Roberto Alvarado', 67], ['Brian Gutiérrez', 7]], 'South Africa': [['Ronwen Williams', 62], ['Thabang Matuludi', 3], ['Khulumani Ndamane', 5], ['Teboho Mokoena', 51], ['Thalente Mbatha', 15], ['Aubrey Modiba', 44], ['Oswin Appollis', 26], ['Tshepang Moremi', 10], ['Lyle Foster', 27], ['Relebohile Mofokeng', 13], ['Themba Zwane', 54], ['Thapelo Maseko', 10], ['Sphephelo Sithole', 28], ['Mbekezeli Mbokazi', 10], ['Iqraam Rayners', 14], ['Sipho Chaine', 4], ['Evidence Makgopa', 26], ['Samukele Kabini', 6], ['Nkosinathi Sibisi', 20], ['Khuliso Mudau', 32], ['Ime Okon', 8], ['Ricardo Goss', 5], ['Jayden Adams', 5], ['Olwethu Makhanya', 1], ['Kamogelo Sebelebele', 3], ['Bradley Cross', 1]], 'South Korea': [['Kim Seung-gyu', 87], ['Lee Han-beom', 8], ['Lee Gi-hyuk', 3], ['Kim Min-jae', 79], ['Kim Tae-hyeon', 7], ['Hwang In-beom', 73], ['Son Heung-min', 144], ['Paik Seung-ho', 27], ['Cho Gue-sung', 44], ['Lee Jae-sung', 105], ['Hwang Hee-chan', 79], ['Song Bum-keun', 3], ['Lee Tae-seok', 15], ['Cho Wi-je', 1], ['Kim Moon-hwan', 35], ['Park Jin-seob', 14], ['Bae Jun-ho', 13], ['Oh Hyeon-gyu', 27], ['Lee Kang-in', 47], ['Yang Hyun-jun', 9], ['Jo Hyeon-woo', 48], ['Seol Young-woo', 34], ['Jens Castrop', 7], ['Kim Jin-gyu', 22], ['Eom Ji-sung', 9], ['Lee Dong-gyeong', 18]], 'Bosnia and Herzegovina': [['Nikola Vasilj', 26], ['Nihad Mujakić', 12], ['Dennis Hadžikadunić', 32], ['Tarik Muharemović', 14], ['Sead Kolašinac', 65], ['Benjamin Tahirović', 28], ['Amar Dedić', 28], ['Armin Gigović', 20], ['Samed Baždar', 13], ['Ermedin Demirović', 40], ['Edin Džeko', 148], ['Mladen Jurkas', 0], ['Ivan Bašić', 17], ['Ivan Šunjić', 11], ['Amar Memić', 13], ['Amir Hadžiahmetović', 36], ['Dženis Burnić', 20], ['Nikola Katić', 17], ['Kerim Alajbegović', 10], ['Esmir Bajraktarević', 16], ['Stjepan Radeljić', 5], ['Martin Zlomislić', 3], ['Haris Tabaković', 10], ['Arjan Malić', 8], ['Jovo Lukić', 3], ['Ermin Mahmić', 2]], 'Canada': [['Dayne St. Clair', 20], ['Alistair Johnston', 58], ['Alfie Jones', 2], ['Luc de Fougerolles', 13], ['Joel Waterman', 17], ['Mathieu Choinière', 23], ['Stephen Eustáquio', 56], ['Ismaël Koné', 40], ['Cyle Larin', 90], ['Jonathan David', 77], ['Liam Millar', 41], ['Tani Oluwaseyi', 24], ['Derek Cornelius', 44], ['Jacob Shaffelburg', 31], ['Moïse Bombito', 20], ['Maxime Crépeau', 32], ['Tajon Buchanan', 60], ['Owen Goodman', 0], ['Alphonso Davies', 58], ['Ali Ahmed', 24], ['Jonathan Osorio', 90], ['Richie Laryea', 75], ['Niko Sigur', 19], ['Promise David', 10], ['Nathan Saliba', 15], ['Jayden Nelson', 14]], 'Qatar': [['Mahmud Abunada', 5], ['Pedro Miguel', 99], ['Lucas Mendes', 25], ['Issa Laye', 4], ['Jassem Gaber', 32], ['Abdulaziz Hatem', 117], ['Ahmed Alaaeldin', 68], ['Edmilson Junior', 16], ['Mohammed Muntari', 67], ['Hassan Al-Haydos', 186], ['Akram Afif', 125], ['Karim Boudiaf', 118], ['Ayoub Al-Oui', 6], ['Homam Ahmed', 68], ['Yusuf Abdurisag', 39], ['Boualem Khoukhi', 116], ['Ahmed Al-Ganehi', 13], ['Sultan Al-Brake', 17], ['Almoez Ali', 115], ['Ahmed Fathy', 48], ['Salah Zakaria', 8], ['Meshaal Barsham', 52], ['Assim Madibo', 51], ['Tahsin Jamshid', 3], ['Al-Hashmi Al-Hussain', 8], ['Mohamed Manai', 10]], 'Switzerland': [['Gregor Kobel', 21], ['Miro Muheim', 10], ['Silvan Widmer', 60], ['Nico Elvedi', 67], ['Manuel Akanji', 81], ['Denis Zakaria', 65], ['Breel Embolo', 86], ['Remo Freuler', 88], ['Johan Manzambi', 12], ['Granit Xhaka', 146], ['Dan Ndoye', 31], ['Yvon Mvogo', 13], ['Ricardo Rodriguez', 138], ['Ardon Jashari', 8], ['Djibril Sow', 52], ['Christian Fassnacht', 23], ['Rubén Vargas', 61], ['Eray Cömert', 22], ['Noah Okafor', 25], ['Michel Aebischer', 40], ['Marvin Keller', 1], ['Fabian Rieder', 28], ['Zeki Amdouni', 29], ['Aurèle Amenda', 7], ['Luca Jaquez', 3], ['Cedric Itten', 15]], 'Brazil': [['Alisson', 78], ['Éderson Silva', 3], ['Gabriel Magalhães', 17], ['Marquinhos', 105], ['Casemiro', 86], ['Alex Sandro', 45], ['Vinícius Júnior', 49], ['Bruno Guimarães', 43], ['Matheus Cunha', 23], ['Neymar', 128], ['Raphinha', 39], ['Weverton', 11], ['Danilo Luiz', 70], ['Bremer', 8], ['Léo Pereira', 4], ['Douglas Santos', 7], ['Fabinho', 33], ['Danilo Santos', 4], ['Endrick', 17], ['Lucas Paquetá', 63], ['Luiz Henrique', 15], ['Gabriel Martinelli', 23], ['Ederson Moraes', 32], ['Roger Ibañez', 7], ['Igor Thiago', 4], ['Rayan', 2]], 'Haiti': [['Johny Placide', 82], ['Carlens Arcus', 56], ['Keeto Thermoncy', 1], ['Ricardo Adé', 59], ['Hannes Delcroix', 7], ['Carl Sainté', 26], ['Derrick Etienne Jr.', 51], ['Martin Expérience', 21], ['Duckens Nazon', 82], ['Jean-Ricner Bellegarde', 10], ['Louicius Deedson', 32], ['Alexandre Pierre', 16], ['Duke Lacroix', 16], ['Garven Metusala', 16], ['Ruben Providence', 15], ['Lenny Joseph', 2], ['Danley Jean Jacques', 31], ['Wilson Isidor', 4], ['Yassin Fortuné', 4], ['Frantzdy Pierrot', 51], ['Josué Casimir', 7], ['Jean-Kévin Duverne', 17], ['Josué Duverger', 7], ['Wilguens Paugain', 8], ['Dominique Simon', 2], ['Woodensky Pierre', 1]], 'Morocco': [['Yassine Bounou', 90], ['Achraf Hakimi', 96], ['Noussair Mazraoui', 45], ['Sofyan Amrabat', 75], ['Marwane Saâdane', 8], ['Ayyoub Bouaddi', 3], ['Chemsdine Talbi', 5], ['Azzedine Ounahi', 49], ['Soufiane Rahimi', 37], ['Brahim Díaz', 26], ['Ismael Saibari', 31], ['Munir Mohamedi', 53], ['Zakaria El Ouahdi', 3], ['Issa Diop', 4], ['Samir El Mourabet', 4], ['Gessime Yassine', 5], ['Amine Sbaï', 2], ['Chadi Riad', 6], ['Youssef Belammari', 19], ['Ayoub El Kaabi', 71], ['Ayoube Amaimouni', 2], ['Ahmed Reda Tagnaouti', 3], ['Bilal El Khannouss', 37], ['Neil El Aynaoui', 16], ['Redouane Halhal', 3], ['Anass Salah-Eddine', 9]], 'Scotland': [['Angus Gunn', 22], ['Aaron Hickey', 21], ['Andy Robertson', 94], ['Scott McTominay', 70], ['Grant Hanley', 68], ['Kieran Tierney', 56], ['John McGinn', 86], ['Tyler Fletcher', 2], ['Lyndon Dykes', 51], ['Ché Adams', 47], ['Ryan Christie', 68], ['Liam Kelly', 3], ['Jack Hendry', 38], ['Ross Stewart', 3], ['John Souttar', 24], ['Dominic Hyam', 4], ['Ben Gannon-Doak', 14], ['George Hirst', 10], ['Lewis Ferguson', 24], ['Lawrence Shankland', 20], ['Craig Gordon', 84], ['Nathan Patterson', 26], ['Kenny McLean', 58], ['Anthony Ralston', 27], ['Findlay Curtis', 3], ['Scott McKenna', 50]], 'Australia': [['Mathew Ryan', 104], ['Miloš Degenek', 57], ['Alessandro Circati', 13], ['Jacob Italiano', 5], ['Jordan Bos', 27], ['Jason Geria', 14], ['Mathew Leckie', 80], ['Connor Metcalfe', 36], ['Mohamed Touré', 10], ['Ajdin Hrustic', 37], ['Awer Mabil', 38], ['Paul Izzo', 4], ["Aiden O'Neill", 31], ['Cammy Devlin', 5], ['Kai Trewin', 6], ['Aziz Behich', 84], ['Nestory Irankunda', 15], ['Patrick Beach', 2], ['Harry Souttar', 38], ['Cristian Volpato', 1], ['Cameron Burgess', 27], ['Jackson Irvine', 82], ['Nishan Velupillay', 7], ['Paul Okon-Engstler', 6], ['Lucas Herrington', 4], ['Tete Yengi', 1]], 'Paraguay': [['Gatito Fernández', 31], ['Gustavo Velázquez', 13], ['Omar Alderete', 36], ['Juan José Cáceres', 17], ['Fabián Balbuena', 48], ['Júnior Alonso', 71], ['Ramón Sosa', 29], ['Diego Gómez', 24], ['Antonio Sanabria', 48], ['Miguel Almirón', 76], ['Maurício', 3], ['Orlando Gill', 6], ['José Canale', 2], ['Andrés Cubas', 33], ['Gustavo Gómez', 89], ['Damián Bobadilla', 19], ['Kaku', 33], ['Álex Arce', 15], ['Julio Enciso', 32], ['Braian Ojeda', 17], ['Gabriel Ávalos', 23], ['Gastón Olveira', 1], ['Matías Galarza', 15], ['Gustavo Caballero', 2], ['Isidro Pitta', 5], ['Alexandro Maidana', 2]], 'Turkey': [['Mert Günok', 37], ['Zeki Çelik', 61], ['Merih Demiral', 62], ['Çağlar Söyüncü', 60], ['Salih Özcan', 30], ['Orkun Kökçü', 50], ['Kerem Aktürkoğlu', 52], ['Arda Güler', 29], ['Deniz Gül', 8], ['Hakan Çalhanoğlu', 105], ['Kenan Yıldız', 28], ['Altay Bayındır', 12], ['Eren Elmalı', 23], ['Abdülkerim Bardakcı', 27], ['Ozan Kabak', 30], ['İsmail Yüksek', 32], ['İrfan Can Kahveci', 47], ['Mert Müldür', 45], ['Yunus Akgün', 19], ['Ferdi Kadıoğlu', 30], ['Barış Alper Yılmaz', 35], ['Kaan Ayhan', 73], ['Uğurcan Çakır', 39], ['Oğuz Aydın', 11], ['Samet Akaydin', 19], ['Can Uzun', 6]], 'United States': [['Matt Turner', 54], ['Sergiño Dest', 39], ['Chris Richards', 36], ['Tyler Adams', 54], ['Antonee Robinson', 54], ['Auston Trusty', 8], ['Giovanni Reyna', 38], ['Weston McKennie', 66], ['Ricardo Pepi', 37], ['Christian Pulisic', 86], ['Brenden Aaronson', 58], ['Miles Robinson', 40], ['Tim Ream', 82], ['Sebastian Berhalter', 13], ['Cristian Roldan', 47], ['Alex Freeman', 17], ['Malik Tillman', 30], ['Max Arfsten', 20], ['Haji Wright', 20], ['Folarin Balogun', 27], ['Timothy Weah', 51], ['Mark McKenzie', 29], ['Joe Scally', 26], ['Matt Freese', 15], ['Chris Brady', 1], ['Alejandro Zendejas', 14]], 'Curaçao': [['Eloy Room', 71], ['Shurandy Sambo', 8], ['Juriën Gaari', 60], ['Roshon van Eijma', 28], ['Sherel Floranus', 28], ['Godfried Roemeratoe', 28], ['Juninho Bacuna', 49], ['Livano Comenencia', 20], ['Jürgen Locadia', 13], ['Leandro Bacuna', 72], ['Jeremy Antonisse', 27], ['Sontje Hansen', 6], ['Tyrese Noslin', 7], ['Kenji Gorré', 38], ["Ar'jany Martha", 9], ['Jearl Margaritha', 22], ['Brandley Kuwas', 35], ['Armando Obispo', 6], ['Gervane Kastaneer', 29], ['Joshua Brenet', 18], ['Tahith Chong', 6], ['Kevin Felida', 19], ['Riechedly Bazoer', 5], ['Deveron Fonville', 2], ['Tyrick Bodak', 4], ['Trevor Doornbusch', 8]], 'Ecuador': [['Hernán Galíndez', 35], ['Félix Torres', 49], ['Piero Hincapié', 52], ['Joel Ordóñez', 17], ['Jordy Alcívar', 11], ['Willian Pacho', 34], ['Pervis Estupiñán', 54], ['Anthony Valencia', 3], ['John Yeboah', 23], ['Kendry Páez', 26], ['Kevin Rodríguez', 31], ['Moisés Ramírez', 7], ['Enner Valencia', 105], ['Alan Minda', 20], ['Pedro Vite', 17], ['Jordy Caicedo', 20], ['Ángelo Preciado', 55], ['Denil Castillo', 5], ['Gonzalo Plata', 50], ['Nilson Angulo', 14], ['Alan Franco', 58], ['Gonzalo Valle', 4], ['Moisés Caicedo', 61], ['Jeremy Arévalo', 4], ['Jackson Porozo', 10], ['Yaimar Medina', 6]], 'Germany': [['Manuel Neuer', 124], ['Antonio Rüdiger', 82], ['Waldemar Anton', 13], ['Jonathan Tah', 47], ['Aleksandar Pavlović', 11], ['Joshua Kimmich', 110], ['Kai Havertz', 58], ['Leon Goretzka', 70], ['Jamie Leweling', 5], ['Jamal Musiala', 42], ['Nick Woltemade', 11], ['Oliver Baumann', 13], ['Pascal Groß', 18], ['Maximilian Beier', 9], ['Nico Schlotterbeck', 27], ['Angelo Stiller', 8], ['Florian Wirtz', 41], ['Nathaniel Brown', 5], ['Leroy Sané', 76], ['Nadiem Amiri', 11], ['Alexander Nübel', 3], ['David Raum', 37], ['Felix Nmecha', 8], ['Malick Thiaw', 5], ['Assan Ouédraogo', 1], ['Deniz Undav', 9]], 'Ivory Coast': [['Yahia Fofana', 35], ['Ousmane Diomande', 15], ['Ghislain Konan', 54], ['Jean Michaël Seri', 65], ['Wilfried Singo', 34], ['Seko Fofana', 32], ['Odilon Kossounou', 35], ['Franck Kessié', 103], ['Ange-Yoan Bonny', 1], ['Simon Adingra', 29], ['Yan Diomande', 10], ['Elye Wahi', 2], ['Christopher Opéri', 12], ['Oumar Diakité', 29], ['Amad Diallo', 19], ['Mohamed Koné', 0], ['Guéla Doué', 20], ['Ibrahim Sangaré', 58], ['Nicolas Pépé', 55], ['Emmanuel Agbadou', 20], ['Evan Ndicka', 28], ['Evann Guessand', 21], ['Alban Lafont', 4], ['Bazoumana Touré', 6], ['Parfait Guiagon', 5], ['Christ Inao Oulaï', 9]], 'Japan': [['Zion Suzuki', 24], ['Yukinari Sugawara', 21], ['Shōgo Taniguchi', 38], ['Kō Itakura', 40], ['Yūto Nagatomo', 145], ['Shuto Machino', 14], ['Ao Tanaka', 38], ['Takefusa Kubo', 49], ['Keisuke Gotō', 4], ['Ritsu Dōan', 65], ['Daizen Maeda', 27], ['Keisuke Ōsako', 11], ['Keito Nakamura', 25], ['Junya Itō', 69], ['Daichi Kamada', 49], ['Tsuyoshi Watanabe', 11], ['Yuito Suzuki', 6], ['Ayase Ueda', 39], ['Kōki Ogawa', 15], ['Ayumu Seko', 14], ['Hiroki Itō', 24], ['Takehiro Tomiyasu', 43], ['Tomoki Hayakawa', 4], ['Kaishū Sano', 13], ['Junnosuke Suzuki', 6], ['Kento Shiogai', 2]], 'Netherlands': [['Bart Verbruggen', 29], ['Lutsharel Geertruida', 21], ['Marten de Roon', 43], ['Virgil van Dijk', 92], ['Nathan Aké', 59], ['Jan Paul van Hecke', 12], ['Justin Kluivert', 12], ['Ryan Gravenberch', 27], ['Wout Weghorst', 52], ['Memphis Depay', 109], ['Cody Gakpo', 50], ['Mats Wieffer', 15], ['Robin Roefs', 1], ['Tijjani Reijnders', 32], ['Micky van de Ven', 21], ['Guus Til', 7], ['Noa Lang', 15], ['Donyell Malen', 53], ['Brian Brobbey', 12], ['Teun Koopmeiners', 28], ['Frenkie de Jong', 66], ['Denzel Dumfries', 72], ['Mark Flekken', 12], ['Crysencio Summerville', 2], ['Jorrel Hato', 8], ['Quinten Timber', 11]], 'Sweden': [['Jacob Widell Zetterström', 3], ['Gustaf Lagerbielke', 11], ['Victor Lindelöf', 76], ['Isak Hien', 29], ['Gabriel Gudmundsson', 24], ['Herman Johansson', 3], ['Lucas Bergvall', 10], ['Daniel Svensson', 13], ['Alexander Isak', 58], ['Benjamin Nygren', 11], ['Anthony Elanga', 30], ['Viktor Johansson', 12], ['Ken Sema', 33], ['Hjalmar Ekdal', 13], ['Carl Starfelt', 18], ['Jesper Karlström', 25], ['Viktor Gyökeres', 33], ['Yasin Ayari', 21], ['Mattias Svanberg', 41], ['Eric Smith', 2], ['Alexander Bernhardsson', 11], ['Besfort Zeneli', 8], ['Kristoffer Nordfeldt', 21], ['Elliot Stroud', 1], ['Gustaf Nilsson', 10], ['Taha Ali', 2]], 'Tunisia': [['Mouhib Chamakh', 3], ['Ali Abdi', 46], ['Montassar Talbi', 64], ['Omar Rekik', 6], ['Adem Arous', 2], ['Dylan Bronn', 52], ['Elias Achouri', 30], ['Elias Saad', 15], ['Hazem Mastouri', 19], ['Hannibal Mejbri', 45], ['Ismaël Gharbi', 17], ['Mortadha Ben Ouanes', 18], ['Rani Khedira', 3], ['Khalil Ayari', 4], ['Hadj Mahmoud', 9], ['Aymen Dahmen', 37], ['Ellyes Skhiri', 83], ['Rayan Elloumi', 4], ['Firas Chaouat', 30], ['Yan Valery', 22], ['Mohamed Amine Ben Hamida', 13], ['Sabri Ben Hessen', 2], ['Moutaz Neffati', 5], ['Raed Chikhaoui', 0], ['Anis Ben Slimane', 41], ['Sebastian Tounekti', 12]], 'Belgium': [['Thibaut Courtois', 109], ['Zeno Debast', 26], ['Arthur Theate', 33], ['Brandon Mechele', 9], ['Maxim De Cuyper', 19], ['Axel Witsel', 138], ['Kevin De Bruyne', 119], ['Youri Tielemans', 85], ['Romelu Lukaku', 126], ['Leandro Trossard', 51], ['Jérémy Doku', 43], ['Senne Lammens', 2], ['Mike Penders', 0], ['Dodi Lukébakio', 30], ['Thomas Meunier', 80], ['Koni De Winter', 8], ['Charles De Ketelaere', 30], ['Joaquin Seys', 5], ['Diego Moreira', 3], ['Hans Vanaken', 34], ['Timothy Castagne', 63], ['Alexis Saelemaekers', 24], ['Nicolas Raskin', 13], ['Amadou Onana', 29], ['Nathan Ngoy', 4], ['Matias Fernandez-Pardo', 2]], 'Egypt': [['Mohamed El Shenawy', 76], ['Yasser Ibrahim', 17], ['Mohamed Hany', 42], ['Hossam Abdelmaguid', 13], ['Ramy Rabia', 44], ['Mohamed Abdelmonem', 36], ['Trézéguet', 96], ['Emam Ashour', 29], ['Hamza Abdelkarim', 2], ['Mohamed Salah', 116], ['Mostafa Ziko', 2], ['Haissem Hassan', 4], ['Ahmed Fatouh', 39], ['Hamdy Fathy', 63], ['Karim Hafez', 9], ['El Mahdy Soliman', 0], ['Mohanad Lasheen', 23], ['Nabil Emad', 12], ['Marwan Attia', 34], ['Ibrahim Adel', 24], ['Mahmoud Saber', 15], ['Omar Marmoush', 50], ['Mostafa Shobeir', 9], ['Tarek Alaa', 3], ['Zizo', 63], ['Mohamed Alaa', 0]], 'Iran': [['Alireza Beiranvand', 86], ['Saleh Hardani', 18], ['Ehsan Hajsafi', 146], ['Shojae Khalilzadeh', 58], ['Milad Mohammadi', 76], ['Saeid Ezatolahi', 83], ['Alireza Jahanbakhsh', 98], ['Mohammad Mohebi', 36], ['Mehdi Taremi', 105], ['Mehdi Ghayedi', 30], ['Ali Alipour', 14], ['Payam Niazmand', 15], ['Hossein Kanaanizadegan', 65], ['Saman Ghoddos', 68], ['Rouzbeh Cheshmi', 40], ['Mehdi Torabi', 52], ['Aria Yousefi', 14], ['Amirhossein Hosseinzadeh', 18], ['Ali Nemati', 17], ['Shahriyar Moghanlou', 21], ['Mohammad Ghorbani', 16], ['Hossein Hosseini', 13], ['Ramin Rezaeian', 74], ['Dennis Eckert', 0], ['Danial Eiri', 0], ['Amirmohammad Razzaghinia', 4]], 'New Zealand': [['Max Crocombe', 24], ['Tim Payne', 51], ['Francis de Vries', 20], ['Tyler Bindon', 25], ['Michael Boxall', 63], ['Joe Bell', 32], ['Matthew Garbett', 37], ['Marko Stamenić', 39], ['Chris Wood', 90], ['Sarpreet Singh', 28], ['Elijah Just', 44], ['Alex Paulsen', 8], ['Liberato Cacace', 37], ['Alex Rufer', 26], ['Nando Pijnaker', 25], ['Finn Surman', 19], ['Kosta Barbarouses', 76], ['Ben Waine', 31], ['Ben Old', 24], ['Callum McCowatt', 32], ['Jesse Randall', 11], ['Michael Woud', 6], ['Ryan Thomas', 25], ['Callan Elliot', 11], ['Lachlan Bayliss', 4], ['Tommy Smith', 56]], 'Cape Verde': [['Vozinha', 90], ['Stopira', 61], ['Diney', 32], ['Roberto Lopes', 45], ['Logan Costa', 28], ['Kevin Pina', 31], ['Jovane Cabral', 29], ['João Paulo', 41], ['Gilson Benchimol', 21], ['Jamiro Monteiro', 55], ['Garry Rodrigues', 61], ['Márcio Rosa', 11], ['Sidny Lopes Cabral', 11], ['Deroy Duarte', 33], ['Laros Duarte', 20], ['Yannick Semedo', 11], ['Willy Semedo', 38], ['Telmo Arcanjo', 16], ['Dailon Livramento', 22], ['Ryan Mendes', 98], ['Nuno da Costa', 9], ['Steven Moreira', 20], ['CJ dos Santos', 1], ['Wagner Pina', 14], ['Kelvin Pires', 6], ['Hélio Varela', 21]], 'Saudi Arabia': [['Nawaf Al-Aqidi', 24], ['Ali Majrashi', 21], ['Ali Lajami', 24], ['Abdulelah Al-Amri', 44], ['Hassan Al-Tambakti', 54], ['Nasser Al-Dawsari', 47], ['Musab Al-Juwayr', 37], ['Ayman Yahya', 26], ['Firas Al-Buraikan', 72], ['Salem Al-Dawsari', 111], ['Saleh Al-Shehri', 59], ['Saud Abdulhamid', 55], ['Nawaf Boushal', 27], ['Hassan Kadesh', 21], ['Abdullah Al-Khaibari', 42], ['Ziyad Al-Johani', 12], ['Khalid Al-Ghannam', 7], ['Alaa Al-Hejji', 3], ['Abdullah Al-Hamdan', 52], ['Sultan Mandash', 7], ['Mohammed Al-Owais', 65], ['Ahmed Al-Kassar', 9], ['Mohamed Kanno', 79], ['Moteb Al-Harbi', 13], ['Jehad Thakri', 8], ['Mohammed Abu Al-Shamat', 8]], 'Spain': [['David Raya', 13], ['Marc Pubill', 2], ['Álex Grimaldo', 14], ['Eric García', 21], ['Marcos Llorente', 24], ['Mikel Merino', 43], ['Ferran Torres', 57], ['Fabián Ruiz', 42], ['Gavi', 30], ['Dani Olmo', 50], ['Yéremy Pino', 23], ['Pedro Porro', 18], ['Joan Garcia', 2], ['Aymeric Laporte', 46], ['Álex Baena', 17], ['Rodri', 62], ['Nico Williams', 30], ['Martín Zubimendi', 26], ['Lamine Yamal', 25], ['Pedri', 41], ['Mikel Oyarzabal', 53], ['Pau Cubarsí', 12], ['Unai Simón', 58], ['Marc Cucurella', 24], ['Víctor Muñoz', 2], ['Borja Iglesias', 8]], 'Uruguay': [['Sergio Rochet', 35], ['José María Giménez', 99], ['Sebastián Cáceres', 24], ['Ronald Araújo', 27], ['Manuel Ugarte', 36], ['Rodrigo Bentancur', 74], ['Nicolás de la Cruz', 34], ['Federico Valverde', 73], ['Darwin Núñez', 38], ['Giorgian de Arrascaeta', 60], ['Facundo Pellistri', 39], ['Santiago Mele', 8], ['Guillermo Varela', 28], ['Agustín Canobbio', 15], ['Emiliano Martínez', 10], ['Mathías Olivera', 35], ['Matías Viña', 43], ['Brian Rodríguez', 32], ['Rodrigo Aguirre', 10], ['Maximiliano Araújo', 28], ['Federico Viñas', 11], ['Joaquín Piquerez', 19], ['Fernando Muslera', 134], ['Santiago Bueno', 8], ['Juan Manuel Sanabria', 5], ['Rodrigo Zalazar', 8]], 'France': [['Brice Samba', 4], ['Malo Gusto', 11], ['Lucas Digne', 58], ['Dayot Upamecano', 38], ['Jules Koundé', 48], ['Manu Koné', 14], ['Ousmane Dembélé', 59], ['Aurélien Tchouaméni', 46], ['Marcus Thuram', 34], ['Kylian Mbappé', 98], ['Michael Olise', 17], ['Bradley Barcola', 20], ["N'Golo Kanté", 69], ['Adrien Rabiot', 59], ['Ibrahima Konaté', 28], ['Mike Maignan', 40], ['William Saliba', 32], ['Warren Zaïre-Emery', 11], ['Théo Hernandez', 44], ['Désiré Doué', 7], ['Lucas Hernandez', 42], ['Jean-Philippe Mateta', 4], ['Robin Risser', 0], ['Rayan Cherki', 7], ['Maghnes Akliouche', 9], ['Maxence Lacroix', 4]], 'Iraq': [['Fahad Talib', 21], ['Rebin Sulaka', 56], ['Hussein Ali', 27], ['Zaid Tahseen', 28], ['Akam Hashim', 14], ['Manaf Younis', 34], ['Youssef Amyn', 27], ['Ibrahim Bayesh', 76], ['Ali Al-Hamadi', 20], ['Mohanad Ali', 72], ['Ahmed Qasem', 3], ['Jalal Hassan', 102], ['Ali Yousif', 7], ['Zidane Iqbal', 25], ['Ahmed Maknzi', 7], ['Amir Al-Ammari', 51], ['Ali Jasim', 36], ['Aymen Hussein', 95], ['Kevin Yakob', 9], ['Aimar Sher', 7], ['Marko Farji', 12], ['Ahmed Basil', 16], ['Merchas Doski', 31], ['Zaid Ismail', 6], ['Mustafa Saadoon', 17], ['Frans Putros', 28]], 'Norway': [['Ørjan Nyland', 71], ['Morten Thorsby', 31], ['Kristoffer Ajer', 52], ['Leo Østigård', 38], ['David Møller Wolfe', 22], ['Patrick Berg', 43], ['Alexander Sørloth', 72], ['Sander Berge', 66], ['Erling Haaland', 50], ['Martin Ødegaard', 68], ['Jørgen Strand Larsen', 28], ['Sander Tangvik', 0], ['Egil Selvik', 7], ['Fredrik Aursnes', 22], ['Fredrik André Bjørkan', 21], ['Marcus Holmgren Pedersen', 32], ['Torbjørn Heggem', 15], ['Kristian Thorstvedt', 37], ['Thelo Aasgaard', 8], ['Antonio Nusa', 24], ['Andreas Schjelderup', 12], ['Oscar Bobb', 20], ['Jens Petter Hauge', 15], ['Sondre Langås', 3], ['Henrik Falchener', 1], ['Julian Ryerson', 43]], 'Senegal': [['Yehvann Diouf', 2], ['Mamadou Sarr', 8], ['Kalidou Koulibaly', 103], ['Abdoulaye Seck', 22], ['Idrissa Gueye', 131], ['Pathé Ciss', 30], ['Assane Diao', 5], ['Lamine Camara', 33], ['Bamba Dieng', 23], ['Sadio Mané', 128], ['Nicolas Jackson', 33], ['Cherif Ndiaye', 19], ['Iliman Ndiaye', 40], ['Ismail Jakobs', 30], ['Krépin Diatta', 61], ['Édouard Mendy', 57], ['Pape Matar Sarr', 40], ['Ismaïla Sarr', 83], ['Moussa Niakhaté', 31], ['Ibrahim Mbaye', 11], ['Habib Diarra', 21], ['Bara Sapoko Ndiaye', 1], ['Mory Diaw', 5], ['Antoine Mendy', 7], ['El Hadji Malick Diouf', 20], ['Pape Gueye', 41]], 'Algeria': [['Melvin Mastil', 1], ['Aïssa Mandi', 117], ['Achref Abada', 7], ['Mohamed Amine Tougai', 29], ['Zineddine Belaïd', 10], ['Ramiz Zerrouki', 52], ['Riyad Mahrez', 114], ['Houssem Aouar', 20], ['Amine Gouiri', 22], ['Farès Chaïbi', 29], ['Anis Hadj Moussa', 14], ['Nadhir Benbouali', 3], ['Jaouen Hadjam', 17], ['Hicham Boudaoui', 32], ['Rayan Aït-Nouri', 28], ['Oussama Benbot', 2], ['Rafik Belghali', 12], ['Mohamed Amoura', 45], ['Nabil Bentaleb', 59], ['Adil Boulbina', 11], ['Ramy Bensebaini', 81], ['Ibrahim Maza', 16], ['Luca Zidane', 7], ['Yacine Titraoui', 5], ['Farès Ghedjemis', 1], ['Samir Chergui', 4]], 'Argentina': [['Juan Musso', 4], ['Marcos Senesi', 3], ['Nicolás Tagliafico', 76], ['Gonzalo Montiel', 39], ['Leandro Paredes', 77], ['Lisandro Martínez', 28], ['Rodrigo De Paul', 87], ['Valentín Barco', 4], ['Julián Alvarez', 51], ['Lionel Messi', 199], ['Giovani Lo Celso', 67], ['Gerónimo Rulli', 8], ['Cristian Romero', 51], ['Exequiel Palacios', 40], ['Nicolás González', 51], ['Thiago Almada', 16], ['Giuliano Simeone', 13], ['Nico Paz', 9], ['Nicolás Otamendi', 132], ['Alexis Mac Allister', 46], ['José Manuel López', 5], ['Lautaro Martínez', 77], ['Emiliano Martínez', 59], ['Enzo Fernández', 42], ['Facundo Medina', 9], ['Nahuel Molina', 58]], 'Austria': [['Alexander Schlager', 26], ['David Affengruber', 1], ['Kevin Danso', 32], ['Xaver Schlager', 51], ['Stefan Posch', 52], ['Nicolas Seiwald', 47], ['Marko Arnautović', 133], ['David Alaba', 113], ['Marcel Sabitzer', 98], ['Florian Grillitsch', 58], ['Michael Gregoritsch', 75], ['Florian Wiegele', 1], ['Patrick Pentz', 18], ['Saša Kalajdžić', 22], ['Philipp Lienhart', 41], ['Phillipp Mwene', 30], ['Carney Chukwuemeka', 3], ['Romano Schmid', 34], ['Dejan Ljubičić', 9], ['Konrad Laimer', 57], ['Patrick Wimmer', 30], ['Alexander Prass', 19], ['Marco Friedl', 11], ['Paul Wanner', 3], ['Michael Svoboda', 4], ['Alessandro Schöpf', 35]], 'Jordan': [['Yazeed Abulaila', 76], ['Mohammad Abu Hashish', 56], ['Abdallah Nasib', 65], ['Husam Abu Dahab', 18], ['Yazan Al-Arab', 80], ['Amer Jamous', 19], ['Mohammad Abu Zrayq', 41], ['Noor Al-Rawabdeh', 68], ['Ali Olwan', 66], ['Musa Al-Taamari', 92], ['Odeh Al-Fakhouri', 10], ['Nour Bani Attiah', 5], ['Mahmoud Al-Mardi', 89], ['Rajaei Ayed', 72], ['Ibrahim Sadeh', 57], ['Mo Abualnadi', 18], ['Salim Obaid', 11], ['Mohammad Taha', 2], ['Saed Al-Rosan', 21], ['Mohannad Abu Taha', 29], ['Nizar Al-Rashdan', 47], ['Abdallah Al-Fakhouri', 11], ['Ihsan Haddad', 92], ['Ali Azaizeh', 4], ['Mohammad Al-Dawoud', 13], ['Anas Badawi', 1]], 'Colombia': [['David Ospina', 130], ['Daniel Muñoz', 46], ['Jhon Lucumí', 37], ['Santiago Arias', 68], ['Kevin Castaño', 25], ['Richard Ríos', 32], ['Luis Díaz', 74], ['Jorge Carrascal', 24], ['Jhon Córdoba', 21], ['James Rodríguez', 126], ['Jhon Arias', 38], ['Camilo Vargas', 42], ['Yerry Mina', 54], ['Gustavo Puerta', 6], ['Juan Portilla', 10], ['Jefferson Lerma', 65], ['Johan Mojica', 45], ['Willer Ditta', 5], ['Cucho Hernández', 9], ['Juan Fernando Quintero', 49], ['Jaminton Campaz', 10], ['Deiver Machado', 15], ['Davinson Sánchez', 79], ['Álvaro Montero', 12], ['Luis Suárez', 12], ['Andrés Gómez', 8]], 'DR Congo': [['Lionel Mpasi', 29], ['Aaron Wan-Bissaka', 12], ['Steve Kapuadi', 4], ['Axel Tuanzebe', 14], ['Dylan Batubinsika', 14], ["Ngal'ayel Mukau", 14], ['Nathanaël Mbuku', 19], ['Samuel Moutoussamy', 58], ['Brian Cipenga', 8], ['Théo Bongonda', 38], ['Gaël Kakuta', 31], ['Joris Kayembe', 26], ['Meschak Elia', 69], ['Noah Sadiki', 20], ['Aaron Tshibola', 17], ['Timothy Fayulu', 3], ['Cédric Bakambu', 70], ['Charles Pickel', 34], ['Fiston Mayele', 37], ['Yoane Wissa', 38], ['Matthieu Epolo', 1], ['Chancel Mbemba', 109], ['Simon Banza', 15], ['Gédéon Kalulu', 28], ['Edo Kayembe', 43], ['Arthur Masuaku', 45]], 'Portugal': [['Diogo Costa', 43], ['Nélson Semedo', 50], ['Rúben Dias', 76], ['Tomás Araújo', 5], ['Diogo Dalot', 35], ['Matheus Nunes', 20], ['Cristiano Ronaldo', 228], ['Bruno Fernandes', 89], ['Gonçalo Ramos', 25], ['Bernardo Silva', 109], ['João Félix', 54], ['José Sá', 5], ['Renato Veiga', 13], ['Gonçalo Inácio', 22], ['João Neves', 22], ['Francisco Trincão', 18], ['Rafael Leão', 44], ['Pedro Neto', 25], ['Gonçalo Guedes', 34], ['João Cancelo', 68], ['Rúben Neves', 67], ['Rui Silva', 3], ['Vitinha', 38], ['Samú Costa', 6], ['Nuno Mendes', 44], ['Francisco Conceição', 17]], 'Uzbekistan': [['Utkir Yusupov', 40], ['Abdukodir Khusanov', 27], ['Khojiakbar Alijonov', 40], ['Farrukh Sayfiev', 46], ['Rustam Ashurmatov', 49], ['Akmal Mozgovoy', 25], ['Otabek Shukurov', 84], ['Jamshid Iskanderov', 38], ['Odiljon Hamrobekov', 72], ['Jaloliddin Masharipov', 74], ['Oston Urunov', 42], ['Abduvohid Nematov', 8], ['Sherzod Nasrullaev', 31], ['Eldor Shomurodov', 92], ['Umar Eshmurodov', 29], ['Botirali Ergashev', 2], ['Dostonbek Khamdamov', 34], ['Abdulla Abdullaev', 17], ['Azizjon Ganiev', 19], ['Azizbek Amonov', 12], ['Igor Sergeev', 83], ['Abbosbek Fayzullaev', 32], ['Sherzod Esanov', 1], ['Bekhruz Karimov', 2], ['Avazbek Ulmasaliev', 0], ['Jakhongir Urozov', 4]], 'Croatia': [['Dominik Livaković', 75], ['Josip Stanišić', 31], ['Marin Pongračić', 20], ['Joško Gvardiol', 48], ['Duje Ćaleta-Car', 38], ['Josip Šutalo', 33], ['Nikola Moro', 10], ['Mateo Kovačić', 113], ['Andrej Kramarić', 116], ['Luka Modrić', 198], ['Ante Budimir', 38], ['Ivor Pandur', 0], ['Nikola Vlašić', 63], ['Ivan Perišić', 154], ['Mario Pašalić', 85], ['Martin Baturina', 19], ['Petar Sučić', 17], ['Kristijan Jakić', 17], ['Toni Fruk', 7], ['Igor Matanović', 9], ['Luka Sučić', 21], ['Luka Vušković', 5], ['Dominik Kotarski', 4], ['Marco Pašalić', 15], ['Martin Erlić', 13], ['Petar Musa', 11]], 'England': [['Jordan Pickford', 84], ['Ezri Konsa', 20], ["Nico O'Reilly", 5], ['Declan Rice', 73], ['John Stones', 89], ['Marc Guéhi', 29], ['Bukayo Saka', 49], ['Elliot Anderson', 9], ['Harry Kane', 114], ['Jude Bellingham', 48], ['Marcus Rashford', 72], ['Tino Livramento', 6], ['Dean Henderson', 5], ['Jordan Henderson', 90], ['Dan Burn', 8], ['Kobbie Mainoo', 14], ['Morgan Rogers', 15], ['Anthony Gordon', 19], ['Ollie Watkins', 22], ['Noni Madueke', 11], ['Eberechi Eze', 17], ['Ivan Toney', 8], ['James Trafford', 2], ['Reece James', 24], ['Djed Spence', 6], ['Jarell Quansah', 3]], 'Ghana': [['Lawrence Ati-Zigi', 29], ['Alidu Seidu', 24], ['Caleb Yirenkyi', 11], ['Jonas Adjetey', 10], ['Thomas Partey', 57], ['Abdul Mumin', 5], ['Abdul Fatawu', 28], ['Kwasi Sibo', 8], ['Jordan Ayew', 120], ['Brandon Thomas-Asante', 8], ['Antoine Semenyo', 34], ['Joseph Anang', 1], ['Christopher Bonsu Baah', 9], ['Gideon Mensah', 40], ['Elisha Owusu', 20], ['Benjamin Asare', 11], ['Abdul Rahman Baba', 51], ['Jerome Opoku', 11], ['Iñaki Williams', 26], ['Augustine Boakye', 0], ['Kojo Peprah Oppong', 4], ['Kamaldeen Sulemana', 28], ['Derrick Luckassen', 1], ['Ernest Nuamah', 18], ['Prince Kwabena Adu', 5], ['Marvin Senaya', 2]], 'Panama': [['Luis Mejía', 56], ['César Blackman', 40], ['José Córdoba', 32], ['Fidel Escobar', 99], ['Edgardo Fariña', 18], ['Cristian Martínez', 66], ['José Luis Rodríguez', 70], ['Adalberto Carrasquilla', 73], ['Tomás Rodríguez', 13], ['Ismael Díaz', 57], ['Yoel Bárcenas', 104], ['César Samudio', 5], ['Jiovany Ramos', 23], ['Carlos Harvey', 28], ['Eric Davis', 107], ['Andrés Andrade', 50], ['José Fajardo', 68], ['Cecilio Waterman', 55], ['Alberto Quintero', 141], ['Aníbal Godoy', 159], ['César Yanis', 56], ['Orlando Mosquera', 48], ['Michael Amir Murillo', 94], ['Azarias Londoño', 11], ['Roderick Miller', 50], ['Jorge Gutiérrez', 18]]}

print('Loaded', len(SQUADS_2026), 'squads,', sum(len(v) for v in SQUADS_2026.values()), 'players')

---
## 3 · Quick EDA

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.2))
results.assign(y=results.date.dt.year).groupby("y").size().plot(ax=ax[0],color="#1f77b4")
ax[0].set(title="Matches per year",xlabel="",ylabel="matches")
g=results.assign(y=results.date.dt.year,goals=results.home_score+results.away_score)
g.groupby("y").goals.mean().rolling(5).mean().plot(ax=ax[1],color="#d62728")
ax[1].set(title="Avg goals/match (5y roll)",xlabel="",ylabel="goals")
plt.tight_layout(); plt.show()
nz=results[~results.neutral]
print(f"Home win {(nz.home_score>nz.away_score).mean():.1%} | Draw {(nz.home_score==nz.away_score).mean():.1%} | Away {(nz.home_score<nz.away_score).mean():.1%}")

---
## 4 · Elo Rating Engine (built from scratch, leak-free)

$$R' = R + K\,G\,(W - W_e),\qquad W_e=\frac{1}{1+10^{-(\Delta+H)/400}}$$

K = match importance, G = goal-difference multiplier, H = home advantage (+100, 0 at
neutral). We log the **pre-match** gap + outcome of every game → the leak-free training
table for the ML models.

In [ ]:
def k_factor(t):
    t=str(t).lower()
    if "world cup" in t and "qualif" not in t: return 60
    if "confederations" in t: return 50
    if any(x in t for x in ["uefa euro","copa am","african cup","afc asian","gold cup","nations league"]): return 40
    if "qualif" in t: return 40
    if "friendly" in t: return 20
    return 30
def g_mult(gd):
    gd=abs(gd)
    if gd<=1: return 1.0
    if gd==2: return 1.5
    return (11+gd)/8.0
HOME_ADV=100
elo={};
def get(t): return elo.get(t,1500.0)
hist=[]; elo_hist={}
for r in results.itertuples(index=False):
    h,a=r.home_team,r.away_team; Rh,Ra=get(h),get(a); ha=0 if r.neutral else HOME_ADV
    We=1/(1+10**(-((Rh-Ra)+ha)/400))
    Wh=1.0 if r.home_score>r.away_score else (0.5 if r.home_score==r.away_score else 0.0)
    if h in elo and a in elo: hist.append((r.date,h,a,Rh,Ra,ha,r.home_score,r.away_score,Wh))
    d=k_factor(r.tournament)*g_mult(r.home_score-r.away_score)*(Wh-We)
    elo[h]=Rh+d; elo[a]=Ra-d
    elo_hist.setdefault(h,[]).append((r.date,elo[h])); elo_hist.setdefault(a,[]).append((r.date,elo[a]))
train=pd.DataFrame(hist,columns=["date","home","away","Rh","Ra","ha","hs","as_","Wh"])
print(f"Ratings for {len(elo):,} teams | training rows {len(train):,}")
print("Top WC contenders by Elo:")
print(pd.Series({t:round(get(t)) for t in WC_TEAMS}).sort_values(ascending=False).head(10).to_string())

### 4.1 Validation — does Elo predict outcomes?

In [ ]:
v=train[train.date>="2015-01-01"].copy(); v["gap"]=(v.Rh-v.Ra)+v.ha
v["hw"]=(v.hs>v.as_).astype(int)
print(v.groupby(pd.cut(v.gap,[-1000,-200,-100,0,100,200,1000])).hw.mean().round(3).to_string())
acc=(((v.gap>0)&(v.hs>v.as_))|((v.gap<0)&(v.hs<v.as_))).mean()
print(f"Elo directional accuracy: {acc:.1%}")

---
## 5 · Building the Team Feature Matrix (the 48 finalists)

We assemble every measurable signal for each qualified team. Each becomes a column we
both **analyse** (Section 6) and **feed into the model** (Section 8). Squad-quality
features are built from the **real 26-man squads** (Section 2.2): each player is matched
to his EA FC 26 record. Where FC 26 can identify at least 12 of a team's 26 we use the
actual squad (42 teams); for the few it under-covers we fall back to that nation's
player pool and flag it. **Experience (caps) is taken from the real squad for all 48.**

In [ ]:
LATEST=results.date.max()
def elo_trend(t,months=24):
    h=elo_hist.get(t,[]);
    if not h: return 0.0
    cut=LATEST-pd.Timedelta(days=30*months); past=[v for d,v in h if d<=cut]
    return h[-1][1]-(past[-1] if past else h[0][1])
def recent_form(t,n=10):
    m=results[(results.home_team==t)|(results.away_team==t)].tail(n)
    if len(m)==0: return 0.0
    p=0
    for r in m.itertuples(index=False):
        gf,ga=(r.home_score,r.away_score) if r.home_team==t else (r.away_score,r.home_score)
        p+= 3 if gf>ga else (1 if gf==ga else 0)
    return p/(3*len(m))

# ---- EA FC 26 squad metrics ----
# Two fixes vs a naive approach:
#  (1) The dataset includes WOMEN's players (Liga F, NWSL). For a men's World Cup we
#      drop them so squad ratings aren't contaminated.
#  (2) FC 26 has NO single 'Overall' column. A flat mean of the 6 face stats would
#      punish specialists (a striker's low Defending drags the mean) and reward
#      all-rounders — e.g. it ranked Uruguay's Valverde above Mbappé. Instead we build
#      a POSITION-WEIGHTED overall ('ovr') so a striker is judged on attacking traits,
#      a defender on defensive ones. This reproduces a realistic star order
#      (Mbappé, Dembélé, Vini Jr., Salah, Bellingham...).
att6=["Pace","Shooting","Passing","Dribbling","Defending","Physicality"]
for c in att6: fc26[c]=pd.to_numeric(fc26[c],errors="coerce")
WOMENS_LEAGUES={"Liga F Moeve","NWSL"}
fc26=fc26[~fc26["League"].isin(WOMENS_LEAGUES)].copy()       # men only

POS_W={
 "ATT":dict(Pace=.18,Shooting=.30,Passing=.12,Dribbling=.25,Defending=.00,Physicality=.15),
 "WNG":dict(Pace=.18,Shooting=.20,Passing=.20,Dribbling=.27,Defending=.03,Physicality=.12),
 "MID":dict(Pace=.08,Shooting=.15,Passing=.27,Dribbling=.20,Defending=.18,Physicality=.12),
 "DEF":dict(Pace=.16,Shooting=.00,Passing=.14,Dribbling=.05,Defending=.40,Physicality=.25),
}
_ATT={"ST","CF","LW","RW"}; _WNG={"LM","RM","CAM"}; _MID={"CM","CDM"}; _DEF={"CB","LB","RB","LWB","RWB"}
def _pos_group(p):
    if p in _ATT: return "ATT"
    if p in _WNG: return "WNG"
    if p in _MID: return "MID"
    if p in _DEF: return "DEF"
    return "GK"
def _ovr(row):
    g=_pos_group(row["Position"])
    if g=="GK":                                   # face stats don't describe keepers
        return np.nanmean([row.get("Reactions"),row.get("Composure")])
    w=POS_W[g]; return sum(row[a]*w[a] for a in att6)
fc26["ovr"]=fc26.apply(_ovr,axis=1)
fc26["atk"]=fc26[["Shooting","Dribbling","Pace","Passing"]].mean(axis=1)
fc26["dfn"]=fc26[["Defending","Physicality"]].mean(axis=1)
TOP5={"Premier League","LALIGA EA SPORTS","Serie A Enilive","Bundesliga","Ligue 1 McDonald's"}
ATK_POS={"ST","LW","RW","CAM","LM","RM","CF"}; DEF_POS={"CB","LB","RB","CDM","LWB","RWB"}

# ---- Match each player in the REAL 26-man squad to FC 26 -----------------------------
# We now use the actual official squads (Section 2.2) instead of a best-23 proxy.
# FC 26 covers most European/American squads well but under-represents some Asian/African
# domestic leagues, so where it can identify <12 of a team's 26 we fall back to the
# nation player-pool proxy and flag it. Experience (caps) comes from the real squad for
# ALL 48 teams regardless, since caps are in the squad data itself.
import unicodedata, re as _re
def _norm(s):
    s=unicodedata.normalize("NFKD",str(s)).encode("ascii","ignore").decode().lower()
    return _re.sub(r"\s+"," ",_re.sub(r"[^a-z ]"," ",s)).strip()
fc26=fc26.reset_index(drop=True); fc26["_n"]=fc26["Name"].map(_norm)
_by_full={}; _by_sur={}
for _i in fc26.index:
    _by_full.setdefault(fc26.at[_i,"_n"],_i); _tk=fc26.at[_i,"_n"].split()
    if _tk: _by_sur.setdefault((_tk[-1],_tk[0][0]),[]).append(_i)
def _match(name):
    nm=_norm(name)
    if nm in _by_full: return _by_full[nm]
    tk=nm.split()
    if not tk: return None
    c=_by_sur.get((tk[-1],tk[0][0]),[])
    if len(c)==1: return c[0]
    sur=[i for i in fc26.index if fc26.at[i,"_n"].split() and fc26.at[i,"_n"].split()[-1]==tk[-1]]
    return sur[0] if len(sur)==1 else None

MIN_MATCH=12
def squad_block(team):
    squad=SQUADS_2026.get(team,[])
    caps_mean=float(np.mean([c for _,c in squad])) if squad else np.nan   # real-squad caps
    idx=[_match(n) for n,_ in squad]; idx=[i for i in idx if i is not None]
    if len(idx)>=MIN_MATCH:
        sub=fc26.loc[idx].copy(); source="real_squad"          # the ACTUAL 26
    else:
        sub=fc26[fc26.canon==team].copy(); source="proxy"      # FC26 can't ID this squad
    if len(sub)==0:
        return dict(exp_caps=caps_mean, squad_source="none", nmatch=len(idx))
    best=sub.nlargest(min(23,len(sub)),"ovr")
    atk=sub[sub.Position.isin(ATK_POS)].nlargest(6,"atk")["atk"].mean()
    dfn=sub[sub.Position.isin(DEF_POS)].nlargest(6,"dfn")["dfn"].mean()
    return dict(squad_rating=best["ovr"].mean(), star_rating=sub["ovr"].max(),
                star_dependency=sub["ovr"].max()-best["ovr"].mean(),
                top3_rating=sub.nlargest(3,"ovr")["ovr"].mean(),
                squad_depth=int((sub["ovr"]>=75).sum()), atk_rating=atk, dfn_rating=dfn,
                top5_league_pct=100*best["League"].isin(TOP5).mean(),
                exp_caps=caps_mean, squad_source=source, nmatch=len(idx))
# ---- squad age & market value (Transfermarkt national_teams) ----
tm_val=nat_teams.groupby("canon")["total_market_value"].max()
tm_age=nat_teams.groupby("canon")["average_age"].mean()
# ---- World Cup pedigree ----
wc=results[results.tournament=="FIFA World Cup"]   # finals only (excludes qualification)
def wc_ped(team):
    m=wc[(wc.home_team==team)|(wc.away_team==team)]
    if len(m)==0: return (0.0,0)
    w=sum(1 for r in m.itertuples(index=False)
          if ((r.home_score>r.away_score) if r.home_team==team else (r.away_score>r.home_score)))
    return (w/len(m),len(m))

rows=[]
for t in WC_TEAMS:
    sb=squad_block(t)
    wr,wm=wc_ped(t)
    rows.append(dict(team=t,conf=CONF[t],is_host=int(t in HOSTS),
        elo=get(t),elo_trend=elo_trend(t),form_pts=recent_form(t),
        squad_age=tm_age.get(t,np.nan),
        squad_value=tm_val.get(t,np.nan),wc_winrate=wr,wc_matches=wm,**sb))
feat=pd.DataFrame(rows)
# graceful fallbacks for thin-coverage nations (ratings only; caps are real for all 48)
for c in ["squad_rating","star_rating","top3_rating","atk_rating","dfn_rating",
          "top5_league_pct","squad_depth","exp_caps","squad_age","squad_value","star_dependency"]:
    feat[c]=feat[c].fillna(feat[c].median())
feat["squad_value_log"]=np.log10(feat["squad_value"].clip(lower=1))
feat["conf_strength"]=feat["conf"].map(feat.groupby("conf").elo.mean())
feat=feat.sort_values("elo",ascending=False).reset_index(drop=True)
print("Feature matrix shape:",feat.shape)
print("Squad data source:", feat.squad_source.value_counts().to_dict())
print("Teams on nation-pool fallback (FC26 under-covers them):",
      feat[feat.squad_source!='real_squad'].team.tolist())
feat[["team","elo","squad_rating","star_rating","exp_caps","top5_league_pct","squad_source"]].head(10)

---
## 6 · Deep-Dive: The Factors That Decide Tournaments

Here we examine each driver the question asked about, quantify it from the data, and
note how it should influence the model. The measurable ones are folded into the rating
in Section 8.

### 6.1 Star-player impact & dependency

Two different things matter: the **ceiling** a world-class player gives a team, and the
**risk** of leaning too heavily on one man over a 104-match, month-long tournament.

> **Rating note.** EA FC 26 ships no single "Overall" column, so we build a
> **position-weighted** rating (`ovr`): a striker is judged on attacking traits, a
> defender on defensive ones. A naive flat mean of the six face stats would punish
> specialists — a striker's low Defending would sink it — and wrongly rank an all-round
> midfielder above an elite forward. We also exclude women's-league players, who are
> present in the raw file. With these fixes the star order reads as expected
> (Mbappé, Dembélé, Vini Jr., Salah, Bellingham…). Ceiling = best player's `ovr`;
> dependency = star `ovr` − squad best-23 average.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
s=feat.sort_values("star_rating",ascending=False).head(15).iloc[::-1]
ax[0].barh(s.team,s.star_rating,color="#6a4c93")
ax[0].set(title="Best-player rating (EA FC 26) — top 15",xlabel="rating")
d=feat.sort_values("star_dependency",ascending=False).head(15).iloc[::-1]
ax[1].barh(d.team,d.star_dependency,color="#c1121f")
ax[1].set(title="Star dependency (star − squad avg) — most reliant",xlabel="rating gap")
plt.tight_layout(); plt.show()
print("Highest ceiling :", ", ".join(feat.nlargest(5,'star_rating').team))
print("Most dependent  :", ", ".join(feat.nlargest(5,'star_dependency').team),
      "  <- high upside but fragile if the star is injured/marked out")

**Read:** elite squads (Spain, France, Brazil, Argentina, England) combine a high star
ceiling with low dependency — strength is spread across the team. Several outsiders rely
heavily on a single talisman; that raises their upside in a one-off game but makes them
fragile across a long tournament (which connects directly to the injury limitation in
Section 12). In the model, star/top-3 quality **adds** rating; extreme dependency carries
a small **penalty**.

### 6.2 Squad experience — international caps & age

Tournament football rewards composure. We measure experience as the **mean international
caps of the actual 26-man squad** (straight from the official squad lists — authentic for
all 48 teams) and look at the age profile (peak ≈ 27–29).

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
e=feat.sort_values("exp_caps",ascending=False).head(15).iloc[::-1]
ax[0].barh(e.team,e.exp_caps,color="#1d3557")
ax[0].set(title="Mean caps of top-23 (experience) — top 15",xlabel="caps")
ax[1].scatter(feat.squad_age,feat.elo,s=28,color="#457b9d")
for _,r in feat.sort_values('elo',ascending=False).head(8).iterrows():
    ax[1].annotate(r.team,(r.squad_age,r.elo),fontsize=7)
ax[1].axvspan(27,29,color="green",alpha=.08)
ax[1].set(title="Squad age vs Elo (peak band shaded)",xlabel="avg squad age",ylabel="Elo")
plt.tight_layout(); plt.show()
print("Most experienced:", ", ".join(feat.nlargest(5,'exp_caps').team))

**Read:** the deepest-tournament teams tend to cluster in the 27–29 peak band with high
caps. Very young sides have upside but historically wobble in knockouts; very old sides
risk fatigue late in a long tournament. Experience is folded into the rating as a modest
positive.

### 6.3 League pedigree — quality of weekly competition

A squad whose players compete weekly in Europe's top-5 leagues arrives battle-hardened.
We measure the share of each team's **actual 26-man squad** playing in the Premier
League, LaLiga, Serie A, Bundesliga or Ligue 1 (for teams FC 26 covers; the few
fallback teams use their nation pool).

In [ ]:
l=feat.sort_values("top5_league_pct",ascending=False).head(18).iloc[::-1]
fig,ax=plt.subplots(figsize=(8,5.5))
ax.barh(l.team,l.top5_league_pct,color="#2a9d8f")
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title="Share of squad in a top-5 European league",xlabel="% of squad")
plt.tight_layout(); plt.show()
print("Most top-5-based:", ", ".join(feat.nlargest(5,'top5_league_pct').team))
print("Correlation top5%% vs Elo: %.2f"%feat[["top5_league_pct","elo"]].corr().iloc[0,1])

**Read:** league pedigree correlates strongly with Elo and is one of the better separators of contenders from minnows. Folded in as a positive.

### 6.4 Attack vs defence balance

Where does each team's threat come from? We aggregate FC 26 attacking (best forwards)
and defensive (best defenders) ratings. Balanced, high-on-both teams travel best in
knockouts where one bad half ends your tournament.

In [ ]:
fig,ax=plt.subplots(figsize=(7.5,6))
ax.scatter(feat.dfn_rating,feat.atk_rating,s=30,color="#e76f51")
for _,r in feat.sort_values("elo",ascending=False).head(12).iterrows():
    ax.annotate(r.team,(r.dfn_rating,r.atk_rating),fontsize=7)
ax.axvline(feat.dfn_rating.median(),color="grey",ls="--",alpha=.5)
ax.axhline(feat.atk_rating.median(),color="grey",ls="--",alpha=.5)
ax.set(title="Attack vs Defence (EA FC 26 best lines)",xlabel="defence rating",ylabel="attack rating")
plt.tight_layout(); plt.show()

**Read:** top-right quadrant = elite both ways (the genuine favourites). Attack and defence ratings flow into the squad-quality term of the adjustment.

### 6.5 World-Cup pedigree (past tournament data)

History isn't destiny, but knockout know-how is real. We compute each nation's all-time
World-Cup match win-rate and experience (matches played) from `results.csv`.

In [ ]:
w=feat[feat.wc_matches>=10].sort_values("wc_winrate",ascending=False).head(15).iloc[::-1]
fig,ax=plt.subplots(figsize=(8,5.5))
ax.barh(w.team,w.wc_winrate*100,color="#c9a227")
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title="All-time World-Cup win rate (≥10 WC matches)",xlabel="win %")
plt.tight_layout(); plt.show()
print("Best WC pedigree:", ", ".join(feat.nlargest(5,'wc_winrate').team))
print("Debutants / minimal WC history:",
      ", ".join(feat[feat.wc_matches<=3].team.tolist()))

**Read:** Brazil, Germany, Argentina, Spain, France carry the strongest knockout pedigree; debutants and thin-history sides get no pedigree credit. Folded in as a positive.

### 6.6 Coach impact — an honest data limitation

The question asked about coaching. The Transfermarkt `national_teams.csv` **`coach_name`
column is entirely empty**, and no other authorised dataset contains coach identity,
tenure or managerial track record. Rather than invent a coach-quality score (which would
be fabrication), we **do not model coaching**. It is listed in Section 12 as a known,
unmeasured factor. *To add it properly you would attach a verified coaches dataset (e.g.
manager win-rate / tournament history) and repeat the Section-8 folding step.*

In [ ]:
print("national_teams.coach_name non-null:", nat_teams['coach_name'].notna().sum(), "of", len(nat_teams))
print("-> coaching cannot be quantified from the authorised data; treated as a limitation.")

---
## 7 · Match-Outcome Models

Classifiers train on the **leak-free** historical table (pre-match Elo gap, home
advantage, era). Target = Home / Draw / Away. Compared by accuracy and log-loss.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score

ml=train[train.date>="2005-01-01"].copy()
ml["gap"]=ml.Rh-ml.Ra; ml["elo_sum"]=(ml.Rh+ml.Ra)/2
ml["y"]=np.select([ml.hs>ml.as_,ml.hs==ml.as_],[2,1],0)
X=ml[["gap","ha","elo_sum"]].values; y=ml.y.values
cut=int(len(ml)*0.8); Xtr,Xte,ytr,yte=X[:cut],X[cut:],y[:cut],y[cut:]
sc=StandardScaler().fit(Xtr); Xtr_s,Xte_s=sc.transform(Xtr),sc.transform(Xte)
mods={}
mods["Logistic Regression"]=(LogisticRegression(max_iter=1000,multi_class="multinomial").fit(Xtr_s,ytr),True)
mods["Random Forest"]=(RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=50,random_state=42,n_jobs=-1).fit(Xtr,ytr),False)
try:
    from xgboost import XGBClassifier
    mods["XGBoost"]=(XGBClassifier(n_estimators=350,max_depth=4,learning_rate=.05,subsample=.9,
        colsample_bytree=.9,eval_metric="mlogloss",random_state=42,n_jobs=-1).fit(Xtr,ytr),False)
except Exception as e: print("XGBoost unavailable (Kaggle has it):",e)
print(f"{'Model':22s}{'Acc':>8s}{'LogLoss':>10s}")
sctab={}
for n,(m,s) in mods.items():
    p=m.predict_proba(Xte_s if s else Xte); sctab[n]=(accuracy_score(yte,p.argmax(1)),log_loss(yte,p,labels=[0,1,2]))
    print(f"{n:22s}{sctab[n][0]:>8.3f}{sctab[n][1]:>10.3f}")
best_name=min(sctab,key=lambda k:sctab[k][1]); best_model,best_scaled=mods[best_name]
print("Best:",best_name)

### 7.1 Poisson goal model (for scorelines)

In [ ]:
fm=train[train.date>="2000-01-01"].copy()
dh=(fm.Rh-fm.Ra)+fm.ha; da=(fm.Ra-fm.Rh)-fm.ha
delta=np.r_[dh.values,da.values]/100.0; goals=np.r_[fm.hs.values,fm.as_.values]
b1,b0=np.polyfit(delta,np.log(goals+0.5),1)
def exp_goals(d): return max(0.15,math.exp(b0+b1*d/100.0))
print(f"mu=exp({b0:.3f}+{b1:.3f}*delta/100) | +100 edge -> {exp_goals(100):.2f} vs {exp_goals(-100):.2f}")

---
## 8 · Folding the Factors Into the Rating

Elo captures *results* but lags signals like a suddenly-talented squad, deep top-5-league
representation or thin tournament experience. We convert the measurable factors into a
**capped Elo adjustment** and use the adjusted rating everywhere downstream.

**Transparent weights** (each factor z-scored across the 48 teams):

| Factor | Weight | Direction |
|---|---|---|
| Squad rating (FC 26 best-23) | +0.26 | better squad ↑ |
| Top-5 league share | +0.20 | more elite minutes ↑ |
| Star / top-3 ceiling | +0.16 | higher ceiling ↑ |
| Experience (caps) | +0.12 | more caps ↑ |
| World-Cup win rate | +0.12 | pedigree ↑ |
| Elo momentum (trend) | +0.08 | rising form ↑ |
| Squad depth | +0.06 | deeper bench ↑ |
| Star **dependency** | −0.10 | over-reliance ↓ (risk) |

The weighted z-score is scaled to Elo points and **capped at ±60** so it refines rather
than overrides the 150-year results-based Elo. This is an explicit modelling judgement,
shown with full sensitivity below.

In [ ]:
def z(s):
    s=s.astype(float); return (s-s.mean())/(s.std(ddof=0)+1e-9)
W={"squad_rating":.26,"top5_league_pct":.20,"top3_rating":.16,"exp_caps":.12,
   "wc_winrate":.12,"elo_trend":.08,"squad_depth":.06,"star_dependency":-.10}
feat["factor_index"]=sum(z(feat[k])*w for k,w in W.items())
SCALE, CAP = 42, 60
feat["elo_adjust"]=(SCALE*feat["factor_index"]).clip(-CAP,CAP).round(1)
feat["elo_adjusted"]=(feat["elo"]+feat["elo_adjust"]).round(1)

ELO_ADJ=dict(zip(feat.team,feat.elo_adjust))
def rating(t): return get(t)+ELO_ADJ.get(t,0.0)   # adjusted rating for WC teams only

print("Biggest UPWARD adjustments (squad/Elo lag):")
print(feat.nlargest(6,"elo_adjust")[["team","elo","elo_adjust","elo_adjusted"]].to_string(index=False))
print("\nBiggest DOWNWARD adjustments:")
print(feat.nsmallest(6,"elo_adjust")[["team","elo","elo_adjust","elo_adjusted"]].to_string(index=False))

### 8.1 Before vs after — how the factors move each team

In [ ]:
top=feat.sort_values("elo_adjusted",ascending=False).head(16)
fig,ax=plt.subplots(figsize=(9,6)); ypos=np.arange(len(top))
ax.hlines(ypos,top.elo,top.elo_adjusted,color="grey",alpha=.5)
ax.scatter(top.elo,ypos,label="Elo (results only)",color="#999",zorder=3)
ax.scatter(top.elo_adjusted,ypos,label="Adjusted (factors folded in)",color="#e63946",zorder=3)
ax.set_yticks(ypos); ax.set_yticklabels(top.team); ax.invert_yaxis()
ax.set(title="Rating before vs after folding in the factors",xlabel="rating"); ax.legend()
plt.tight_layout(); plt.show()

### 8.2 Unified match predictor (uses the **adjusted** rating)

In [ ]:
from scipy.stats import poisson
def _raw_wdl(rh,ra,ha):
    x=np.array([[rh-ra,ha,(rh+ra)/2]]); xx=sc.transform(x) if best_scaled else x
    return best_model.predict_proba(xx)[0]      # [P_away, P_draw, P_home]
def wdl(home,away,neutral=True,host_edge=0.0):
    Rh,Ra=rating(home),rating(away); ha=(0 if neutral else HOME_ADV)+host_edge
    # At a NEUTRAL venue neither side is truly 'home'. The classifier learned home
    # advantage via the ha feature, so labelling one team 'home' with ha=0 would bias
    # the result. We symmetrise by averaging both orderings so the prediction is
    # order-independent (a real fix that also de-biases the group predictions).
    if neutral and host_edge==0:
        p1=_raw_wdl(Rh,Ra,0); p2=_raw_wdl(Ra,Rh,0)
        return {"home":(p1[2]+p2[0])/2,"draw":(p1[1]+p2[1])/2,"away":(p1[0]+p2[2])/2}
    p=_raw_wdl(Rh,Ra,ha); return {"home":p[2],"draw":p[1],"away":p[0]}
def score(home,away,neutral=True,host_edge=0.0,mx=8):
    Rh,Ra=rating(home),rating(away); ha=(0 if neutral else HOME_ADV)+host_edge
    mh,ma=exp_goals((Rh-Ra)+ha),exp_goals((Ra-Rh)-ha)
    M=np.outer(poisson.pmf(np.arange(mx+1),mh),poisson.pmf(np.arange(mx+1),ma))
    i,j=np.unravel_index(M.argmax(),M.shape); return int(i),int(j),mh,ma
def predict_match(home,away,neutral=True,host_edge=0.0):
    p=wdl(home,away,neutral,host_edge); gh,ga,mh,ma=score(home,away,neutral,host_edge)
    win=home if p["home"]>max(p["away"],p["draw"]) else (away if p["away"]>max(p["home"],p["draw"]) else "Draw")
    return dict(home=home,away=away,p_home=p["home"],p_draw=p["draw"],p_away=p["away"],
                xg_home=round(mh,2),xg_away=round(ma,2),scoreline=f"{gh}-{ga}",predicted=win)
import pprint; pprint.pprint(predict_match("Spain","France"))

---
## 9 · Predicting all 72 Group-Stage Matches
Hosts get a partial home edge in their own country.

In [ ]:
HOST_EDGE=40
def he(h,a): return HOST_EDGE if h in HOSTS else (-HOST_EDGE if a in HOSTS else 0)
rows=[]
for g,ts in GROUPS.items():
    for h,a in itertools.combinations(ts,2):
        pm=predict_match(h,a,neutral=True,host_edge=he(h,a)); pm["group"]=g; rows.append(pm)
group_pred=pd.DataFrame(rows)[["group","home","away","p_home","p_draw","p_away","xg_home","xg_away","scoreline","predicted"]]
group_pred[["p_home","p_draw","p_away"]]=group_pred[["p_home","p_draw","p_away"]].round(3)
group_pred.to_csv("wc2026_group_match_predictions.csv",index=False)
print(f"Predicted & saved {len(group_pred)} group matches.")
group_pred[group_pred.group=="H"]

---
## 10 · Monte-Carlo Tournament Simulation (10,000 runs)

Every match (group + knockout) is resolved by the **adjusted-rating** Poisson model;
groups ranked on points → GD → goals; top 2 + best 8 third-places advance; knockouts are
single-elimination with Elo-weighted shootouts.

> **Bracket note (explicit approximation).** FIFA's official R32 third-place allocation
> has 495 predefined combinations. We instead seed the 32 qualifiers by strength tier
> (winners > runners-up > thirds), preserving the real "winners get easier early draws"
> principle. Championship odds are robust to this because every match is still resolved
> by the calibrated model.

In [ ]:
def seed_order(n):
    o=[1,2]
    while len(o)<n:
        m=len(o)*2+1; o=[x for s in o for x in (s,m-s)]
    return o
SEED_ORDER32=seed_order(32)   # standard bracket: keeps the strongest seeds apart

def sim_goals(h,a,host_edge=0.0):
    Rh,Ra=rating(h),rating(a)
    return RNG.poisson(exp_goals((Rh-Ra)+host_edge)),RNG.poisson(exp_goals((Ra-Rh)-host_edge))
def wp(a,b): return 1/(1+10**(-(rating(a)-rating(b))/400))
def ko(a,b):
    gh,ga=sim_goals(a,b,he(a,b))
    if gh>ga: return a
    if ga>gh: return b
    return a if RNG.random()<wp(a,b) else b
def sim_group(ts):
    pts={t:0 for t in ts}; gf={t:0 for t in ts}; ga={t:0 for t in ts}
    for h,a in itertools.combinations(ts,2):
        x,y=sim_goals(h,a,he(h,a)); gf[h]+=x;ga[h]+=y;gf[a]+=y;ga[a]+=x
        if x>y:pts[h]+=3
        elif y>x:pts[a]+=3
        else:pts[h]+=1;pts[a]+=1
    rk=sorted(ts,key=lambda t:(pts[t],gf[t]-ga[t],gf[t],RNG.random()),reverse=True)
    return rk,pts,{t:gf[t]-ga[t] for t in ts},gf
def simulate():
    win=[];run=[];thd=[]
    for g,ts in GROUPS.items():
        rk,pts,gd,gf=sim_group(ts); win.append(rk[0]);run.append(rk[1])
        thd.append((rk[2],pts[rk[2]],gd[rk[2]],gf[rk[2]]))
    best=[t[0] for t in sorted(thd,key=lambda x:(x[1],x[2],x[3],RNG.random()),reverse=True)[:8]]
    qual=win+run+best
    reached={t:"Group" for t in WC_TEAMS}
    for t in qual: reached[t]="R32"
    tier={**{t:0 for t in win},**{t:1 for t in run},**{t:2 for t in best}}
    ranked=sorted(qual,key=lambda t:(tier[t],-rating(t)))   # seed 1 = strongest
    bs=[ranked[s-1] for s in SEED_ORDER32]                  # standard bracket seeding
    rnd=[(bs[2*k],bs[2*k+1]) for k in range(16)]            # top seeds kept apart
    stages=["R16","QF","SF","Final","Champion"]; si=0
    while True:
        wn=[ko(a,b) for a,b in rnd]
        for w in wn:
            if si<len(stages): reached[w]=stages[si]
        if len(wn)==1: reached[wn[0]]="Champion"; return reached
        rnd=[(wn[i],wn[i+1]) for i in range(0,len(wn),2)]; si+=1
_=simulate(); print("single sim OK")

In [ ]:
N_SIMS=10000
order=["Champion","Final","SF","QF","R16","R32","Group"]; rv={s:i for i,s in enumerate(order)}
cnt={t:{s:0 for s in order} for t in WC_TEAMS}
for _ in range(N_SIMS):
    rc=simulate()
    for t,s in rc.items():
        for s2 in order[rv[s]:]: cnt[t][s2]+=1
prob=pd.DataFrame(cnt).T
for c in order: prob[c]/=N_SIMS
prob=prob.rename(columns={"Champion":"win_cup","Final":"reach_final","SF":"reach_SF",
                          "QF":"reach_QF","R16":"reach_R16","R32":"reach_R32"})
prob["elo_adj"]=[rating(t) for t in prob.index]
prob=prob.sort_values("win_cup",ascending=False)
print(f"Ran {N_SIMS:,} sims.")
prob[["win_cup","reach_final","reach_SF","reach_QF","elo_adj"]].head(15).round(3)

### 10.1 Win-probability — top 20

In [ ]:
fig,ax=plt.subplots(figsize=(8,6)); top=prob.head(20).iloc[::-1]
ax.barh(top.index,top.win_cup*100,color="#e76f51")
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title=f"P(win WC 2026) — {N_SIMS:,} sims, factor-adjusted",xlabel="win prob")
for i,v in enumerate(top.win_cup*100): ax.text(v+.1,i,f"{v:.1f}%",va="center",fontsize=8)
plt.tight_layout(); plt.show()

---
## 11 · Final Predictions

In [ ]:
final=prob.copy()
for c in ["win_cup","reach_final","reach_SF"]: final[c+"_pct"]=(final[c]*100).round(1)
out=final[["win_cup_pct","reach_final_pct","reach_SF_pct","elo_adj"]].reset_index().rename(columns={"index":"team"})
out.to_csv("wc2026_tournament_probabilities.csv",index=False)
print("Saved -> wc2026_tournament_probabilities.csv")
print(f"\n>>> PICK TO WIN: {final.index[0]} ({final.iloc[0].win_cup:.1%})")
print(f">>> Finalists  : {list(final.index[:2])}")
gw=pd.DataFrame([(g,max(ts,key=lambda t:prob.loc[t,'reach_R32']),
                  round(prob.loc[max(ts,key=lambda t:prob.loc[t,'reach_R32']),'reach_R32']*100,1))
                 for g,ts in GROUPS.items()],columns=["group","likely_top","advance_%"])
gw

### 11.2 The predicted tournament bracket

A single **most-likely path**: we take each group's expected table (from win/draw/loss
probabilities), seed the 32 qualifiers into a standard bracket (top seeds kept apart),
and let the more probable team win each tie through to the trophy. This is one
deterministic storyline — distinct from the win-probability table above, which averages
over all 10,000 simulated paths — so the bracket champion can differ from the highest
win-% team when one favourite draws a tougher route.

In [ ]:
from matplotlib.patches import FancyBboxPatch
def _hedge(h,a): return HOST_EDGE if h in HOSTS else (-HOST_EDGE if a in HOSTS else 0)
def expected_table(teams):
    pts={t:0.0 for t in teams}; gd={t:0.0 for t in teams}
    for h,a in itertools.combinations(teams,2):
        he2=_hedge(h,a); p=wdl(h,a,True,he2)
        pts[h]+=3*p["home"]+p["draw"]; pts[a]+=3*p["away"]+p["draw"]
        _,_,mh,ma=score(h,a,True,he2); gd[h]+=mh-ma; gd[a]+=ma-mh
    return sorted(teams,key=lambda t:(pts[t],gd[t]),reverse=True),pts,gd
_win=[];_run=[];_thd=[]
for g,ts in GROUPS.items():
    rk,pts,gd=expected_table(ts); _win.append(rk[0]);_run.append(rk[1]);_thd.append((rk[2],pts[rk[2]],gd[rk[2]]))
_best=[t[0] for t in sorted(_thd,key=lambda x:(x[1],x[2]),reverse=True)[:8]]
_qual=_win+_run+_best
_tier={**{t:0 for t in _win},**{t:1 for t in _run},**{t:2 for t in _best}}
_ranked=sorted(_qual,key=lambda t:(_tier[t],-rating(t)))
_bs=[_ranked[s-1] for s in SEED_ORDER32]
def det_winner(a,b):
    p=wdl(a,b,True,_hedge(a,b)); return a if p["home"]>=p["away"] else b
def det_score(a,b):
    gh,ga,_,_=score(a,b,True,_hedge(a,b)); return gh,ga
rounds=[[(_bs[2*k],_bs[2*k+1]) for k in range(16)]]
while True:
    wins=[det_winner(a,b) for a,b in rounds[-1]]
    if len(wins)==1: champion=wins[0]; break
    rounds.append([(wins[i],wins[i+1]) for i in range(0,len(wins),2)])
runner_up=[t for t in rounds[-1][0] if t!=champion][0]

NAVY="#0b1f3a"; GOLD="#c9a227"; WIN="#1b7837"; GREY="#8a8a8a"
COLS=["Round of 32","Round of 16","Quarter-finals","Semi-finals","Final"]
fig,ax=plt.subplots(figsize=(15.5,12.3)); ax.axis("off"); ax.set_xlim(0,16); ax.set_ylim(0,18)
BW,BH=2.55,0.42
def _box(x,y,team,won,sc=None,champ=False):
    fc=GOLD if champ else ("#eaf4ea" if won else "white")
    ec=GOLD if champ else (WIN if won else "#cfcfcf")
    ax.add_patch(FancyBboxPatch((x,y-BH/2),BW,BH,boxstyle="round,pad=0.02,rounding_size=0.06",
                 fc=fc,ec=ec,lw=2.2 if(won or champ) else .9,zorder=3))
    ax.text(x+0.12,y,team,va="center",ha="left",fontsize=8.6,
            fontweight="bold" if(won or champ) else "normal",color=NAVY,zorder=4)
    if sc is not None: ax.text(x+BW-0.12,y,str(sc),va="center",ha="right",fontsize=8.4,
            color=WIN if won else GREY,fontweight="bold",zorder=4)
gap=17.0/16; ypos=[[(i+0.5)*gap for i in range(16)][::-1]]
for r in range(1,len(rounds)):
    pv=ypos[-1]; ypos.append([(pv[2*k]+pv[2*k+1])/2 for k in range(len(rounds[r]))])
xcol=[0.2,3.2,6.2,9.2,12.2]
for r in range(len(rounds)-1):
    for k in range(len(rounds[r+1])):
        y1,y2,yc=ypos[r][2*k],ypos[r][2*k+1],ypos[r+1][k]
        xf,xt=xcol[r]+BW,xcol[r+1]; xm=(xf+xt)/2
        for yy in (y1,y2): ax.plot([xf,xm],[yy,yy],color="#bcbcbc",lw=.8,zorder=1)
        ax.plot([xm,xm],[y1,y2],color="#bcbcbc",lw=.8,zorder=1)
        ax.plot([xm,xt],[yc,yc],color="#bcbcbc",lw=.8,zorder=1)
for r,matches in enumerate(rounds):
    wins=[det_winner(a,b) for a,b in matches]
    for k,(a,b) in enumerate(matches):
        yc=ypos[r][k]; sa,sb=det_score(a,b)
        _box(xcol[r],yc+BH*0.62,a,wins[k]==a,sa); _box(xcol[r],yc-BH*0.62,b,wins[k]==b,sb)
yc=ypos[-1][0]
ax.plot([xcol[-1]+BW,xcol[-1]+BW+1.2],[yc,yc],color=GOLD,lw=2,zorder=2)
_box(xcol[-1]+BW+1.2,yc,f"★ {champion}",False,champ=True)
ax.text(xcol[-1]+BW+1.2+BW/2,yc+0.8,"PREDICTED CHAMPION",ha="center",fontsize=10,fontweight="bold",color=GOLD)
for r,name in enumerate(COLS): ax.text(xcol[r]+BW/2,17.5,name,ha="center",fontsize=11,fontweight="bold",color=NAVY)
fig.suptitle("FIFA World Cup 2026 — Predicted Tournament Bracket",fontsize=17,fontweight="bold",color=NAVY,y=0.99)
fig.text(0.5,0.945,f"Final: {champion} def. {runner_up}",ha="center",fontsize=12,color=GOLD,fontweight="bold")
ax.text(0.2,-0.3,"Model's most-likely path (deterministic). Knockout seeding approximates FIFA's official third-place table.",fontsize=7.5,color=GREY)
plt.tight_layout(rect=[0,0,1,0.93])
plt.savefig("wc2026_predicted_bracket.png",dpi=150,bbox_inches="tight",facecolor="white")
plt.show()
print(f"Predicted champion: {champion} | Runner-up: {runner_up}")
print("Saved -> wc2026_predicted_bracket.png")

---
## 12 · Methodology, Factor Findings & Honest Limitations

**Factors that move the model (measured & folded in — Section 8)**
- **Star power / ceiling** — best & top-3 EA FC 26 ratings (positive); **star dependency**
  carries a small penalty for over-reliance.
- **Squad experience** — mean international caps of the **actual 26-man squad** (positive); age profile shown.
- **League pedigree** — share of squad in a top-5 European league (strong positive,
  high correlation with Elo).
- **Attack/defence balance** — best forward & defender lines from FC 26.
- **Past World-Cup data** — all-time WC win-rate & matches (positive pedigree term).
- **Momentum & depth** — 24-month Elo trend and squad depth.

**Factors NOT modelled (stated honestly, per the data we have)**
- **Coach impact** — `coach_name` is empty in the data and no authorised dataset carries
  manager identity/track record. Not invented. *Add a verified coaches dataset and repeat
  Section 8 to include it.*
- **Injuries** — there is no forward-looking injury feed; over a month-long tournament
  attrition is real but unpredictable from this data. The **star-dependency** metric is
  the closest proxy (teams reliant on one player are most exposed to a single injury).
- **Weather / venue climate** — 2026 spans hot southern US/Mexican venues and temperate
  northern ones; no climate field exists in the data, so it is not modelled. Hosts do get
  a venue edge.

**Other honest caveats**
- **Real squads, with a documented fallback.** Squad features use the official 26-man
  lists, but EA FC 26 can identify only ≥12 players for 42 of the 48 teams; the other
  6 (mostly Asian/African domestic-league sides) fall back to the nation player pool for
  *ratings* (caps are still real for all 48). Printed in Section 5.
- The **knockout bracket seeding is an approximation** of FIFA's 495-combination
  third-place table (Section 10).
- EA FC 26 ratings are a *proxy* for current quality and don't capture very recent form
  or injuries.
- The Elo adjustment is a transparent, **capped (±60)** judgement layered on a 150-year
  results-based rating — not a separately back-tested model (FC 26 squad data only exists
  for 2026, so these squad features can't be validated historically).
- Football is high-variance: even the favourite typically sits under ~25%. These are
  probabilities, **not certainties** — sanity-check against your own football judgement.

**Reproducibility** — seed 42; run top-to-bottom. Outputs:
`wc2026_group_match_predictions.csv`, `wc2026_tournament_probabilities.csv`.

*Sources: FIFA Final Draw (5 Dec 2025); martj42 International Results; davidcariboo
Transfermarkt; talhademirezen FC 26 Player Stats; hubertsidorowicz FBref 2025-26.*